# GPT-2 Training Notebook
**Kaggle / Colab friendly** — FineWeb-Edu streamed, single GPU, auto dtype.

| Section | Cell |
|---|---|
| 1. Install & Imports | #1 – #2 |
| 2. Platform Setup | #3 |
| 3. Data Preparation (FineWeb Streaming) | #4 |
| 4. Model Definition | #5 |
| 5. Training Config | #6 |
| 6. Training Loop | #7 |
| 7. HellaSwag Evaluation | #8 |
| 8. Text Generation | #9 |

In [ ]:
# ── Cell 1 | Install Dependencies ──────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "tiktoken", "datasets", "transformers", "huggingface_hub", "tqdm"])

In [ ]:
# ── Cell 2 | Imports ────────────────────────────────────────────────────────
import os, math, time, json, inspect, requests
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from dataclasses import dataclass
from tqdm import tqdm
from datasets import load_dataset
from transformers import GPT2LMHeadModel

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"bfloat16 : {torch.cuda.is_bf16_supported()}")

In [ ]:
# ── Cell 3 | Platform Setup ─────────────────────────────────────────────────
# Auto-detects Colab / Kaggle and sets paths accordingly.

IS_COLAB  = 'google.colab' in str(globals().get('__builtins__', ''))
IS_KAGGLE = os.path.exists('/kaggle')

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/gpt2_train'
elif IS_KAGGLE:
    BASE_DIR = '/kaggle/working/gpt2_train'
else:                              # local
    BASE_DIR = './gpt2_train'

DATA_DIR = os.path.join(BASE_DIR, 'edu_fineweb')
LOG_DIR  = os.path.join(BASE_DIR, 'log')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(LOG_DIR,  exist_ok=True)

print(f"Platform : {'Colab' if IS_COLAB else 'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Data dir : {DATA_DIR}")
print(f"Log  dir : {LOG_DIR}")

In [ ]:
# ── Cell 4 | FineWeb-Edu Streaming Data Prep ────────────────────────────────
# Streams FineWeb-Edu and writes only MAX_SHARDS shards to disk.
# 1 shard = 100M tokens ≈ 200MB on disk.
# MAX_SHARDS=2 → 1 val shard + 1 train shard ≈ 400MB total.

SHARD_SIZE = int(1e8)   # 100M tokens per shard
MAX_SHARDS = 2          # increase to 4/6 if you have disk space

enc = tiktoken.get_encoding('gpt2')
EOT = enc._special_tokens['<|endoftext|>']

def tokenize(doc):
    tokens = [EOT] + enc.encode_ordinary(doc['text'])
    arr = np.array(tokens, dtype=np.uint16)
    assert arr.min() >= 0 and arr.max() < 2**16
    return arr

def shard_path(shard_idx):
    split = 'val' if shard_idx == 0 else 'train'
    return os.path.join(DATA_DIR, f'edufineweb_{split}_{shard_idx:06d}.npy')

# Check if shards already exist — skip download if so
existing = [shard_path(i) for i in range(MAX_SHARDS)
            if os.path.exists(shard_path(i))]

if len(existing) == MAX_SHARDS:
    print(f'All {MAX_SHARDS} shards already exist — skipping download.')
else:
    print(f'Streaming FineWeb-Edu (sample-10BT) — will write {MAX_SHARDS} shards...')
    fw = load_dataset(
        'HuggingFaceFW/fineweb-edu',
        name='sample-10BT',
        split='train',
        streaming=True          # ← never downloads full dataset
    )

    shard_idx   = 0
    buf         = np.empty((SHARD_SIZE,), dtype=np.uint16)
    token_count = 0
    pbar        = None

    for doc in fw:
        if shard_idx >= MAX_SHARDS:
            break

        tokens = tokenize(doc)

        if token_count + len(tokens) < SHARD_SIZE:
            buf[token_count : token_count + len(tokens)] = tokens
            token_count += len(tokens)
            if pbar is None:
                pbar = tqdm(total=SHARD_SIZE, unit='tok',
                            desc=f'Shard {shard_idx} ({"val" if shard_idx==0 else "train"})')
            pbar.update(len(tokens))
        else:
            remainder = SHARD_SIZE - token_count
            buf[token_count : token_count + remainder] = tokens[:remainder]
            np.save(shard_path(shard_idx), buf)
            print(f'  Saved: {shard_path(shard_idx)}')
            shard_idx  += 1
            if pbar: pbar.close(); pbar = None
            leftover    = tokens[remainder:]
            buf[:len(leftover)] = leftover
            token_count = len(leftover)

    # Write last partial shard
    if token_count > 0 and shard_idx < MAX_SHARDS:
        np.save(shard_path(shard_idx), buf[:token_count])
        print(f'  Saved: {shard_path(shard_idx)}')

    print('Done. Shards on disk:')
    for i in range(MAX_SHARDS):
        p = shard_path(i)
        if os.path.exists(p):
            print(f'  {p}  ({os.path.getsize(p)/1e6:.0f} MB)')

In [ ]:
# ── Cell 5 | GPT-2 Model Definition ─────────────────────────────────────────

# ── Causal Self-Attention ────────────────────────────────────────────────────
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn  = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj  = nn.Linear(config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1
        self.n_head  = config.n_head
        self.n_embd  = config.n_embd

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # flash attn
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

# ── MLP ──────────────────────────────────────────────────────────────────────
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu   = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))

# ── Transformer Block ────────────────────────────────────────────────────────
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp  = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

# ── GPT Config ───────────────────────────────────────────────────────────────
@dataclass
class GPTConfig:
    block_size : int = 1024
    vocab_size : int = 50257
    n_layer    : int = 12
    n_head     : int = 12
    n_embd     : int = 768

# ── GPT Model ────────────────────────────────────────────────────────────────
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h   = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f= nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight  # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, 'NANOGPT_SCALE_INIT'):
                std *= (2 * self.config.n_layer) ** -0.5
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.config.block_size
        pos     = torch.arange(0, T, dtype=torch.long, device=idx.device)
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = tok_emb + pos_emb
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @classmethod
    def from_pretrained(cls, model_type):
        assert model_type in {'gpt2', 'gpt2-medium', 'gpt2-large', 'gpt2-xl'}
        print(f'Loading pretrained weights: {model_type}')
        config_args = {
            'gpt2':        dict(n_layer=12, n_head=12, n_embd=768),
            'gpt2-medium': dict(n_layer=24, n_head=16, n_embd=1024),
            'gpt2-large':  dict(n_layer=36, n_head=20, n_embd=1280),
            'gpt2-xl':     dict(n_layer=48, n_head=25, n_embd=1600),
        }[model_type]
        config_args.update(vocab_size=50257, block_size=1024)
        config = GPTConfig(**config_args)
        model  = GPT(config)
        sd     = model.state_dict()
        sd_hf  = GPT2LMHeadModel.from_pretrained(model_type).state_dict()
        transposed = ['attn.c_attn.weight','attn.c_proj.weight',
                      'mlp.c_fc.weight','mlp.c_proj.weight']
        for k in [k for k in sd_hf if not k.endswith(('.attn.masked_bias', '.attn.bias'))]:
            if any(k.endswith(w) for w in transposed):
                with torch.no_grad(): sd[k].copy_(sd_hf[k].T)
            else:
                with torch.no_grad(): sd[k].copy_(sd_hf[k])
        return model

    def configure_optimizers(self, weight_decay, learning_rate, device_type):
        param_dict = {pn: p for pn, p in self.named_parameters() if p.requires_grad}
        decay_params    = [p for n, p in param_dict.items() if p.dim() >= 2]
        no_decay_params = [p for n, p in param_dict.items() if p.dim() < 2]
        optim_groups = [
            {'params': decay_params,    'weight_decay': weight_decay},
            {'params': no_decay_params, 'weight_decay': 0.0},
        ]
        fused_available = 'fused' in inspect.signature(torch.optim.AdamW).parameters
        use_fused = fused_available and device_type == 'cuda'
        optimizer = torch.optim.AdamW(
            optim_groups, lr=learning_rate,
            betas=(0.9, 0.95), eps=1e-8, fused=use_fused
        )
        return optimizer

print('Model classes defined.')

In [ ]:
# ── Cell 6 | Training Configuration ─────────────────────────────────────────

# ── Hardware ──────────────────────────────────────────────────────────────────
device      = 'cuda' if torch.cuda.is_available() else 'cpu'
device_type = 'cuda' if 'cuda' in device else 'cpu'
# Auto-select dtype: bfloat16 on A100, float16 on T4/P100
pt_dtype    = torch.bfloat16 if (device_type=='cuda' and torch.cuda.is_bf16_supported()) \
              else torch.float16
print(f'Device: {device}  |  dtype: {pt_dtype}')

torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)
torch.set_float32_matmul_precision('high')

# ── Batch / Sequence ──────────────────────────────────────────────────────────
# Effective batch = total_batch_size tokens, split into micro-steps.
# Adjust B based on your VRAM:
#   T4  (15GB) → B=4   | P100 (16GB) → B=8   | A100 (40GB) → B=32
B                  = 4       # micro-batch size  ← tune this
T                  = 1024    # sequence length
total_batch_size   = 524288  # 2**19, ~0.5M tokens — keep fixed for LR schedule
grad_accum_steps   = total_batch_size // (B * T)  # auto-calculated

# ── Training Duration ─────────────────────────────────────────────────────────
max_steps    = 50     # 50  = smoke test (~5 min on T4)
                      # 500 = short run   (~50 min on T4)
                      # 19073 = full 1 epoch on 10B tokens
warmup_steps = 10     # scale with max_steps (~715 for full run)
max_lr       = 6e-4
min_lr       = max_lr * 0.1

# ── LR Schedule (cosine with warmup) ─────────────────────────────────────────
def get_lr(it):
    if it < warmup_steps:
        return max_lr * (it + 1) / warmup_steps
    if it > max_steps:
        return min_lr
    decay_ratio = (it - warmup_steps) / (max_steps - warmup_steps)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (max_lr - min_lr)

print(f'Micro-batch B={B}, T={T}')
print(f'Gradient accumulation steps: {grad_accum_steps}')
print(f'Effective batch size: {B * T * grad_accum_steps:,} tokens')
print(f'Training for {max_steps} steps')

In [ ]:
# ── Cell 6b | DataLoader ─────────────────────────────────────────────────────

def load_tokens(filename):
    npt = np.load(filename)
    return torch.tensor(npt.astype(np.int32), dtype=torch.long)

class DataLoaderLite:
    def __init__(self, B, T, split):
        self.B, self.T = B, T
        shards = sorted([
            os.path.join(DATA_DIR, f)
            for f in os.listdir(DATA_DIR) if split in f
        ])
        assert len(shards) > 0, f'No shards found for split={split} in {DATA_DIR}'
        print(f'DataLoader [{split}]: {len(shards)} shard(s)')
        self.shards = shards
        self.reset()

    def reset(self):
        self.current_shard = 0
        self.tokens = load_tokens(self.shards[self.current_shard])
        self.current_pos = 0

    def next_batch(self):
        B, T = self.B, self.T
        buf = self.tokens[self.current_pos : self.current_pos + B * T + 1]
        x = buf[:-1].view(B, T)
        y = buf[1:].view(B, T)
        self.current_pos += B * T
        if self.current_pos + B * T + 1 > len(self.tokens):
            self.current_shard = (self.current_shard + 1) % len(self.shards)
            self.tokens = load_tokens(self.shards[self.current_shard])
            self.current_pos = 0
        return x, y

train_loader = DataLoaderLite(B=B, T=T, split='train')
val_loader   = DataLoaderLite(B=B, T=T, split='val')
print('DataLoaders ready.')

In [ ]:
# ── Cell 7 | Training Loop ───────────────────────────────────────────────────

# ── Build model ───────────────────────────────────────────────────────────────
# vocab_size=50304 is next multiple of 128 after 50257 → faster matmuls
model = GPT(GPTConfig(vocab_size=50304))
model = model.to(device)
# torch.compile disabled for Colab/Kaggle compatibility
# model = torch.compile(model)

optimizer = model.configure_optimizers(
    weight_decay=0.1, learning_rate=max_lr, device_type=device_type
)

log_file = os.path.join(LOG_DIR, 'log.txt')
with open(log_file, 'w') as f:
    pass  # clear log

# ── Loop ──────────────────────────────────────────────────────────────────────
for step in range(max_steps):
    t0       = time.time()
    last_step = (step == max_steps - 1)

    # ── Validation every 50 steps ──────────────────────────────────────────
    if step % 50 == 0 or last_step:
        model.eval()
        val_loader.reset()
        with torch.no_grad():
            val_loss_accum = 0.0
            for _ in range(20):
                x, y = val_loader.next_batch()
                x, y = x.to(device), y.to(device)
                with torch.autocast(device_type=device_type, dtype=pt_dtype):
                    _, loss = model(x, y)
                val_loss_accum += loss.item() / 20
        print(f'[step {step:4d}] val_loss={val_loss_accum:.4f}')
        with open(log_file, 'a') as f:
            f.write(f'{step} val {val_loss_accum:.6f}\n')

    # ── Checkpoint every 500 steps ──────────────────────────────────────────
    if step > 0 and (step % 500 == 0 or last_step):
        ckpt_path = os.path.join(LOG_DIR, f'model_{step:05d}.pt')
        torch.save({
            'model': model.state_dict(),
            'config': model.config,
            'step': step,
            'val_loss': val_loss_accum,
        }, ckpt_path)
        print(f'  Checkpoint saved → {ckpt_path}')

    # ── Training step ──────────────────────────────────────────────────────
    model.train()
    optimizer.zero_grad()
    loss_accum = 0.0

    for micro_step in range(grad_accum_steps):
        x, y = train_loader.next_batch()
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device_type, dtype=pt_dtype):
            _, loss = model(x, y)
        loss = loss / grad_accum_steps
        loss_accum += loss.detach()
        loss.backward()

    norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    lr   = get_lr(step)
    for pg in optimizer.param_groups:
        pg['lr'] = lr
    optimizer.step()

    if device_type == 'cuda':
        torch.cuda.synchronize()

    dt  = time.time() - t0
    tps = (B * T * grad_accum_steps) / dt
    print(f'step {step:4d} | loss={loss_accum.item():.4f} | '
          f'lr={lr:.2e} | norm={norm:.2f} | '
          f'{dt*1000:.0f}ms | {tps:.0f} tok/s')
    with open(log_file, 'a') as f:
        f.write(f'{step} train {loss_accum.item():.6f}\n')

print('Training complete.')

In [ ]:
# ── Cell 8 | HellaSwag Evaluation ────────────────────────────────────────────
# Run this cell independently to eval any checkpoint.

HELLASWAG_DIR = os.path.join(BASE_DIR, 'hellaswag')
os.makedirs(HELLASWAG_DIR, exist_ok=True)

HELLASWAG_URL = ('https://raw.githubusercontent.com/rowanz/hellaswag'
                 '/master/data/hellaswag_val.jsonl')
HELLASWAG_FILE = os.path.join(HELLASWAG_DIR, 'hellaswag_val.jsonl')

if not os.path.exists(HELLASWAG_FILE):
    print('Downloading HellaSwag val...')
    r = requests.get(HELLASWAG_URL, stream=True)
    with open(HELLASWAG_FILE, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    print('Done.')

def render_example(example):
    ctx    = example['ctx']
    label  = example['label']
    endings = example['endings']
    ctx_tokens = enc.encode(ctx)
    tok_rows, mask_rows = [], []
    for end in endings:
        end_tokens = enc.encode(' ' + end)
        tok_rows.append(ctx_tokens + end_tokens)
        mask_rows.append([0] * len(ctx_tokens) + [1] * len(end_tokens))
    maxlen = max(len(r) for r in tok_rows)
    tokens = torch.zeros(4, maxlen, dtype=torch.long)
    mask   = torch.zeros(4, maxlen, dtype=torch.long)
    for i, (tr, mr) in enumerate(zip(tok_rows, mask_rows)):
        tokens[i, :len(tr)] = torch.tensor(tr)
        mask[i,   :len(mr)] = torch.tensor(mr)
    return tokens, mask, label

def get_most_likely_row(tokens, mask, logits):
    shift_logits = logits[..., :-1, :].contiguous()
    shift_tokens = tokens[..., 1:].contiguous()
    shift_losses = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_tokens.view(-1), reduction='none'
    ).view(tokens.size(0), -1)
    shift_mask = mask[..., 1:].contiguous()
    avg_loss   = (shift_losses * shift_mask).sum(dim=1) / shift_mask.sum(dim=1)
    return avg_loss.argmin().item()

model.eval()
num_correct, num_total = 0, 0

with open(HELLASWAG_FILE, 'r') as f:
    for line in tqdm(f, desc='HellaSwag'):
        example = json.loads(line)
        tokens, mask, label = render_example(example)
        tokens, mask = tokens.to(device), mask.to(device)
        with torch.no_grad():
            with torch.autocast(device_type=device_type, dtype=pt_dtype):
                logits, _ = model(tokens)
        pred_norm = get_most_likely_row(tokens, mask, logits)
        num_correct += int(pred_norm == label)
        num_total   += 1

acc = num_correct / num_total
print(f'HellaSwag accuracy: {num_correct}/{num_total} = {acc:.4f}')
# Baseline — untrained GPT-2 124M ≈ 0.2955 (norm)

In [ ]:
# ── Cell 9 | Text Generation ─────────────────────────────────────────────────

model.eval()
NUM_RETURN_SEQUENCES = 4
MAX_LENGTH           = 64
PROMPT               = 'Hello, I am a language model,'

tokens = enc.encode(PROMPT)
tokens = torch.tensor(tokens, dtype=torch.long)
tokens = tokens.unsqueeze(0).repeat(NUM_RETURN_SEQUENCES, 1).to(device)

sample_rng = torch.Generator(device=device)
sample_rng.manual_seed(42)

with torch.no_grad():
    while tokens.size(1) < MAX_LENGTH:
        with torch.autocast(device_type=device_type, dtype=pt_dtype):
            logits, _ = model(tokens)
        probs     = F.softmax(logits[:, -1, :], dim=-1)
        topk_probs, topk_idx = torch.topk(probs, 50, dim=-1)
        ix   = torch.multinomial(topk_probs, 1, generator=sample_rng)
        xcol = torch.gather(topk_idx, -1, ix)
        tokens = torch.cat([tokens, xcol], dim=1)

for i in range(NUM_RETURN_SEQUENCES):
    decoded = enc.decode(tokens[i].tolist())
    print(f'--- Sample {i+1} ---')
    print(decoded)
    print()